# E1 Raw Baseline — Dual T4

**Setup**
1. Accelerator: GPU T4 x2
2. Attach SeaClear dataset
3. Internet ON (clone + pip)
4. Set `REPO_URL` / `STAGE` below

Stages: `smoke` (2 epochs) → `prep` → `e1` (full baseline).

In [ ]:
# ========= USER CONFIG =========
REPO_URL = "https://github.com/heller007/underwater.git"
REPO_DIR = "/kaggle/working/underwater"  # always this exact path — do not nest
BRANCH = "main"
STAGE = "smoke"  # smoke | prep | e1
HELD_OUT_SITE = "Lokrum"
# Set explicitly if auto-discovery fails:
SEACLEAR_ROOT = None  # e.g. "/kaggle/input/seaclear-marine-debris-detection-and-segmentation"
DEVICE = "0,1"  # dual T4
# ===============================

In [ ]:
import os, subprocess, sys
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU{i}:", torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 1, "Enable GPU T4 x2 accelerator"
if torch.cuda.device_count() < 2:
    print("WARNING: only 1 GPU visible; dual-T4 expected. Continuing with device=0")

In [ ]:
# Install slim deps only (do NOT reinstall torch)
%pip install -q ultralytics imagehash scikit-image pycocotools opencv-python-headless

In [ ]:
from pathlib import Path
import shutil

# Canonical path only — prevents underwater/underwater/underwater nesting
REPO_DIR = "/kaggle/working/underwater"
os.chdir("/kaggle/working")

for nested in (
    Path("/kaggle/working/underwater/underwater"),
    Path("/kaggle/working/underwater/underwater/underwater"),
):
    if nested.exists():
        print("Removing nested clone:", nested)
        shutil.rmtree(nested, ignore_errors=True)

if (Path(REPO_DIR) / ".git").exists():
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--all"])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
elif Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
sys.path = [p for p in sys.path if Path(p).resolve() != Path(REPO_DIR).resolve()]
sys.path.insert(0, REPO_DIR)
print("Repo at", REPO_DIR, "rev", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR).decode().strip())
print("cwd", os.getcwd())
assert not Path(REPO_DIR, "underwater").exists(), "Nested underwater/ still present"


In [ ]:
# Discover / list input datasets — SeaClear MUST be attached via Add Data
from pathlib import Path
from src.common import load_env, hardware_info, list_input_datasets, find_coco_json_under

print("/kaggle/input contents:")
for line in list_input_datasets(Path("/kaggle/input")):
    print(" -", line)

# Auto-pick SEACLEAR_ROOT if user left it as None
if SEACLEAR_ROOT is None:
    env = load_env("kaggle")
    SEACLEAR_ROOT = str(env.seaclear_root) if env.seaclear_root else None

if SEACLEAR_ROOT is None:
    raise SystemExit(
        "SeaClear not found under /kaggle/input.\n"
        "Fix: Notebook → Add Data → search 'Seaclear Marine Debris' → Add.\n"
        "Then re-run, or set SEACLEAR_ROOT = '/kaggle/input/<exact-folder-name>'"
    )

coco = find_coco_json_under(Path(SEACLEAR_ROOT))
print("Using SEACLEAR_ROOT:", SEACLEAR_ROOT)
print("COCO json:", coco)
if coco is None:
    raise SystemExit(f"No COCO annotations JSON under {SEACLEAR_ROOT}")

env = load_env("kaggle")
print("Device:", env.device)
print(hardware_info())
os.environ["SEACLEAR_ROOT"] = SEACLEAR_ROOT

In [ ]:
cmd = [
    sys.executable, "scripts/run_stage.py",
    "--stage", STAGE,
    "--env", "kaggle",
    "--held-out-site", HELD_OUT_SITE,
    "--device", DEVICE if torch.cuda.device_count() >= 2 else "0",
]
if SEACLEAR_ROOT:
    cmd += ["--seaclear-root", SEACLEAR_ROOT]
if STAGE == "smoke":
    cmd += ["--max-images", "100"]

print("Running:", " ".join(cmd))
subprocess.check_call(cmd, cwd=REPO_DIR)

In [ ]:
# List artifacts to download from the notebook Output panel
from pathlib import Path
runs = Path("/kaggle/working/runs")
reports = Path("/kaggle/working/reports")
print("Reports:", list(reports.glob("*"))[:20])
print("Runs:")
for r in sorted(runs.glob("*"))[-5:]:
    print(" ", r)
    for w in r.rglob("*.pt"):
        print("   weight:", w, "size_mb", round(w.stat().st_size/1e6, 2))

## Resume tips

- Re-run the clone cell to pull latest code after a GitHub push.
- Training checkpoints live under `/kaggle/working/runs/.../train/weights/`.
- For long E1 runs, download `best.pt` + `metrics_all.json` before the session expires (~12h).
- Optional: upload a “results” Dataset and copy prior `runs/` into `/kaggle/working` at start to resume.